# Fitxers CSV

FES · UAB · 2026–2027

## Objectius d'aprenentatge

- Representar lectures en una taula amb capçalera.
- Escriure amb `DictWriter` i llegir amb `DictReader`.
- Convertir els valors recuperats i comprovar la capçalera.

Contingut essencial · 30 minuts. Requereix rutes, fitxers i diccionaris.

## Camps, files i capçalera

CSV representa dades tabulars en text. Acordarem els camps `sensor`, `valor` i `unitat` en aquest ordre. La biblioteca `csv` gestiona cometes i comes dins dels valors.

Un contracte de dades compartit permet que el sistema de monitoratge i el lector de fitxers col·laborin.

In [ ]:
import csv
from pathlib import Path
from tempfile import TemporaryDirectory

camps = ["sensor", "valor", "unitat"]
lectures = [
    {"sensor": "T1", "valor": 21.5, "unitat": "°C"},
    {"sensor": "Laboratori, nord", "valor": 48.0, "unitat": "%"},
]
print(lectures[0])


## Files representades amb llistes

`csv.writer` escriu llistes amb `writerow` i `writerows`. `csv.reader` recupera una llista de cadenes per fila. A continuació passarem a `DictWriter` i `DictReader` per accedir als camps pel seu nom.

En obrir el fitxer, `newline=""` deixa que `csv` gestioni els salts de línia.


In [ ]:
with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "tensions.csv"
    with open(ruta, "w", encoding="utf-8", newline="") as fitxer:
        escriptor = csv.writer(fitxer)
        escriptor.writerow(["sensor", "volts"])
        escriptor.writerows([("V1", 3.3), ("V2", 5.0)])
    with open(ruta, encoding="utf-8", newline="") as fitxer:
        print(list(csv.reader(fitxer)))


## Escriure diccionaris

`DictWriter` rep els noms de camp. `writeheader()` escriu la capçalera i `writerows()` escriu els registres. `newline=""` delega a `csv` la gestió dels salts de línia.

Les entrades d'aquest exemple són registres preparats amb valors finits. Observa com s'escriu el nom que conté una coma.

In [ ]:
def desar_csv(ruta, lectures):
    """Desa lectures amb la capçalera acordada."""
    with open(ruta, "w", encoding="utf-8", newline="") as fitxer:
        escriptor = csv.DictWriter(fitxer, fieldnames=camps)
        escriptor.writeheader()
        escriptor.writerows(lectures)

with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "lectures.csv"
    desar_csv(ruta, lectures)
    print(ruta.read_text(encoding="utf-8"))


## Llegir i recuperar els tipus

`DictReader` fa servir la primera fila com a claus. Els valors llegits són cadenes, incloent-hi les mesures.

El càlcul necessita `float(fila["valor"])`. La capçalera exacta ens permet detectar un fitxer d'un altre format. `math.isfinite` rebutja `nan` i `inf`.

In [ ]:
import math


def llegir_csv(ruta):
    """Recupera lectures i comprova capçalera, files i valors."""
    recuperades = []
    with open(ruta, "r", encoding="utf-8", newline="") as fitxer:
        lector = csv.DictReader(fitxer)
        if lector.fieldnames != camps:
            raise ValueError("Capçalera incorrecta")
        for fila in lector:
            if None in fila:
                raise ValueError("La fila ha de tenir tres camps")
            for camp in camps:
                if fila[camp] is None:
                    raise ValueError("La fila ha de tenir tres camps")
            valor = float(fila["valor"])
            if not math.isfinite(valor):
                raise ValueError("La mesura ha de ser finita")
            recuperades.append({
                "sensor": fila["sensor"],
                "valor": valor,
                "unitat": fila["unitat"],
            })
    return recuperades

with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "lectures.csv"
    desar_csv(ruta, lectures)
    recuperades = llegir_csv(ruta)
    print(recuperades)
    print(type(recuperades[0]["valor"]).__name__)  # float


## Taula sense lectures

Una taula sense registres conserva la capçalera. En recuperar-la obtenim `[]`. Un fitxer completament buit incompleix el contracte de capçalera.

In [ ]:
with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "lectures.csv"
    desar_csv(ruta, [])
    print(llegir_csv(ruta))  # []
    ruta.write_text("", encoding="utf-8")
    try:
        llegir_csv(ruta)
    except ValueError as error:
        print(error)


## Experimentar amb un valor invàlid

La validació pertany al límit d'entrada de dades. Substitueix `error` per `25.0` i prediu el nou resultat. El diagnòstic detallat de files malformades és una ampliació.

In [ ]:
with TemporaryDirectory() as carpeta:
    ruta = Path(carpeta) / "lectures.csv"
    ruta.write_text("sensor,valor,unitat\nT1,error,°C\n", encoding="utf-8")
    try:
        llegir_csv(ruta)
    except ValueError:
        print("El valor de la lectura és invàlid")


## Pràctica

Resol els exercicis 1 i 2. El 3 consolida l'anada i tornada i el 4 és un repte opcional de validació.

[Enunciats](02_csv_exercicis.py) · [Solucions](02_csv_solucions.py)

Quin problema apareixeria si el nom del sensor es tractés amb `split(",")`?